## Chuyên viên tri thức

### Tác nhân hỏi đáp đóng vai trò là một chuyên viên tri thức
### Dành cho nhân viên của Insurellm, một công ty công nghệ bảo hiểm
### Tác nhân cần đưa ra câu trả lời chính xác và giải pháp phải có chi phí thấp.

Dự án này sẽ sử dụng RAG (Retrieval Augmented Generation — Sinh tăng cường truy xuất) để đảm bảo trợ lý hỏi đáp của chúng ta có độ chính xác cao.

## HÔM NAY:

- Phần A: Chúng ta sẽ chia tài liệu thành các ĐOẠN
- Phần B: Chúng ta sẽ mã hóa các ĐOẠN thành VECTƠ và lưu vào Chroma
- Phần C: Chúng ta sẽ trực quan hóa các vectơ

### PHẦN A: Chia tài liệu thành các đoạn

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# Giá cả là một yếu tố quan trọng với công ty, vì vậy chúng ta sẽ sử dụng một mô hình có chi phí thấp

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"Khóa API OpenAI tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập khóa API OpenAI")


In [ ]:
# Có bao nhiêu ký tự trong toàn bộ tài liệu?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Tìm thấy {len(files)} tệp trong cơ sở tri thức")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Tổng số ký tự trong cơ sở tri thức: {len(entire_knowledge_base):,}")

In [ ]:
# Có bao nhiêu token trong toàn bộ tài liệu?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Tổng số token cho {MODEL}: {token_count:,}")

In [ ]:
# Nạp toàn bộ nội dung trong cơ sở tri thức bằng các trình nạp của LangChain

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Đã nạp {len(documents)} tài liệu")

In [ ]:
documents[1]

In [ ]:
# Chia thành các đoạn bằng RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Đã chia thành {len(chunks)} đoạn")
print(f"Đoạn đầu tiên:\n\n{chunks[0]}")

In [ ]:
chunks[100]

### PHẦN B: Tạo vectơ và lưu vào Chroma

Ở Tuần 3, bạn đã thiết lập một tài khoản Hugging Face và nhận được `HF_TOKEN`.

Lúc này, bạn có thể thêm nó vào tệp `.env` rồi chạy `load_dotenv(override=True)`.

(Thực ra việc này có lẽ không bắt buộc.)

In [ ]:
# Chọn một mô hình embedding

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Kho vectơ đã được tạo với {vectorstore._collection.count()} tài liệu")

In [ ]:
# Hãy khảo sát các vectơ

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Có {count:,} vectơ với {dimensions:,} chiều trong kho vectơ")

### Phần C: Trực quan hóa!

In [ ]:
# Chuẩn bị

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# Con người chúng ta thấy việc trực quan hóa mọi thứ trong không gian 2D dễ hơn!
# Giảm số chiều của các vectơ xuống 2D bằng t-SNE
# (phép nhúng láng giềng ngẫu nhiên phân phối t)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ phân tán 2D
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='Trực quan hóa kho vectơ Chroma trong không gian 2D',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Hãy thử với không gian 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ phân tán 3D
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Trực quan hóa kho vectơ Chroma trong không gian 3D',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()